<a href="https://colab.research.google.com/github/3430-dotcom/KOAI/blob/main/halla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import math
import warnings
import matplotlib

try:
    import google.colab
    IN_COLAB = True
    from IPython.display import display as _ip_display, Image as _IPImage
except ImportError:
    IN_COLAB = False
    _ip_display = _IPImage = None

if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm

if IN_COLAB:
    import subprocess
    subprocess.run(['apt-get', '-y', 'install', 'fonts-nanum'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    _fm.fontManager.__init__()

_nanum = next((f.fname for f in _fm.fontManager.ttflist if 'Nanum' in f.name), None)
if _nanum:
    plt.rcParams['font.family'] = _fm.FontProperties(fname=_nanum).get_name()

plt.rcParams.update({
    'axes.unicode_minus' : False,
    'figure.facecolor'   : 'white',
    'axes.facecolor'     : '#F7F9FC',
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.grid'          : True,
    'grid.color'         : '#CCCCCC',
    'grid.linestyle'     : ':',
    'grid.alpha'         : 0.6,
    'font.size'          : 12,
    'axes.titlesize'     : 14,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 12,
    'legend.fontsize'    : 11,
    'xtick.labelsize'    : 10,
    'ytick.labelsize'    : 10,
    'lines.linewidth'    : 2,
})

_C = {
    'train'     : '#1565C0',
    'val'       : '#C62828',
    'temperate' : '#2E7D32',
    'cool'      : '#1565C0',
    'subalpine' : '#6A1B9A',
    'scenarios' : ['#1565C0', '#2E7D32', '#E65100', '#B71C1C'],
}


def _show(path: str):
    if IN_COLAB and _ip_display is not None:
        _ip_display(_IPImage(filename=path))

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if IN_COLAB:
    DATA_PATH  = '/content/halla_aws_extreme_all.csv'
    OUTPUT_DIR = '/content/outputs_fixed'
else:
    DATA_PATH  = os.path.join(os.getcwd(), 'halla_aws_extreme_all.csv')
    OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs_fixed')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"  실행 환경: {'Google Colab' if IN_COLAB else '로컬'} | 디바이스: {DEVICE}")

BASELINE_YEAR = 2018
WARMING_RATE  = 0.035

print("="*70)
print("  한라산 고도별 기후대 예측 AI (데이터 누수 수정 버전)")
print("  Time Series Transformer + PINN")
print("="*70)

print("\n[1단계] 데이터 로드 및 전처리")
print("-"*70)

df = pd.read_csv(DATA_PATH)
print(f"원본 데이터: {df.shape[0]}행 x {df.shape[1]}열")

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear

stations = df[['station_id','name','altitude_m']].drop_duplicates().sort_values('altitude_m')
for _, row in stations.iterrows():
    print(f"  - {row['name']} (고도: {row['altitude_m']}m)")

fill_cols = ['altitude_m', 't_min', 't_max', 'rh_min', 'rh_max', 'wind_speed_max', 'rain_daily', 'wind_dir_deg']
for col in fill_cols:
    if col in df.columns:
        df[col] = df.groupby(['station_id', 'month'])[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df.groupby('station_id')[col].transform(lambda x: x.fillna(x.median()))
df = df.dropna(subset=['t_min', 't_max', 'rh_min', 'rh_max'])

df['t_mean'] = (df['t_min'] + df['t_max']) / 2.0
df['t_range'] = df['t_max'] - df['t_min']
df['rh_mean'] = (df['rh_min'] + df['rh_max']) / 2.0

df['es_tmean'] = 6.1078 * np.exp((17.27 * df['t_mean']) / (df['t_mean'] + 237.3))
df['e_actual'] = (df['rh_mean'] / 100.0) * df['es_tmean']
df['e_actual'] = df['e_actual'].clip(lower=0.01)
ln_ratio = np.log(df['e_actual'] / 6.1078)
df['dew_point'] = (237.3 * ln_ratio) / (17.27 - ln_ratio)

df['altitude_norm'] = (df['altitude_m'] - df['altitude_m'].min()) / (df['altitude_m'].max() - df['altitude_m'].min())
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
df['wind_speed_max'] = df['wind_speed_max'].fillna(0)
df['rain_daily'] = df['rain_daily'].fillna(0)
df['wind_dir_deg'] = df['wind_dir_deg'].fillna(0)
df['wind_dir_sin'] = np.sin(np.radians(df['wind_dir_deg']))
df['wind_dir_cos'] = np.cos(np.radians(df['wind_dir_deg']))

_year_span = max(1, df['year'].max() - BASELINE_YEAR)
df['year_norm']  = (df['year'] - BASELINE_YEAR) / _year_span
df['year_trend'] = (df['year'] - BASELINE_YEAR) * WARMING_RATE

def classify_climate_by_altitude(altitude_m):
    if altitude_m < 1200:
        return 0
    elif altitude_m < 1500:
        return 1
    else:
        return 2

df['climate_zone'] = df['altitude_m'].apply(classify_climate_by_altitude)

print(f"\n[FIX-6] 기후대 분류 (고도 기반 외부 기준):")
for z, name in {0:'온대 낙엽활엽수림대 (<1200m)', 1:'냉온대 침엽수림대 (1200-1500m)', 2:'아고산대 (>1500m)'}.items():
    cnt = (df['climate_zone']==z).sum()
    print(f"  Class {z} ({name}): {cnt}개 ({cnt/len(df)*100:.1f}%)")

input_feature_cols = [
    't_min', 't_max',
    't_range',
    'rh_min', 'rh_max',
    'dew_point',
    'wind_speed_max',
    'rain_daily',
    'wind_dir_sin', 'wind_dir_cos',
    'month_sin', 'month_cos',
    'doy_sin', 'doy_cos',
    'year_norm',
    'year_trend',
]

target_cols = ['t_mean', 'rh_mean', 'es_tmean']

overlap = set(input_feature_cols) & set(target_cols)
print(f"\n[FIX-1] 피처-타겟 겹침 확인: {overlap if overlap else '없음 (OK)'}")
print(f"  입력 피처 ({len(input_feature_cols)}개): {input_feature_cols}")
print(f"  타겟 ({len(target_cols)}개): {target_cols}")

print(f"\n[FIX-4] 시간 순서 기반 데이터 분할:")

df_sorted = df.sort_values(['station_id', 'date']).reset_index(drop=True)

train_mask = df_sorted['year'] <= 2023
val_mask = df_sorted['year'] == 2024
test_mask = df_sorted['year'] == 2025

print(f"  학습 (2018-2023): {train_mask.sum()}행")
print(f"  검증 (2024):      {val_mask.sum()}행")
print(f"  테스트 (2025):    {test_mask.sum()}행")

SEQ_LENGTH = 14

features_all = df_sorted[input_feature_cols].values
targets_all = df_sorted[target_cols].values
labels_all = df_sorted['climate_zone'].values
years_all = df_sorted['year'].values
station_ids = df_sorted['station_id'].values

scaler_features = StandardScaler()
scaler_targets = StandardScaler()

train_indices = np.where(train_mask.values)[0]
scaler_features.fit(features_all[train_indices])
scaler_targets.fit(targets_all[train_indices])

features_scaled = scaler_features.transform(features_all)
targets_scaled = scaler_targets.transform(targets_all)

print(f"\n[FIX-3] Scaler: 학습 데이터({len(train_indices)}행)에서만 fit")

def create_sequences(features_scaled, targets_scaled, labels_all, years_all, station_ids, seq_length):
    train_X, train_yr, train_yc = [], [], []
    val_X, val_yr, val_yc = [], [], []
    test_X, test_yr, test_yc = [], [], []

    for sid in np.unique(station_ids):
        idx = np.where(station_ids == sid)[0]
        sf = features_scaled[idx]
        st = targets_scaled[idx]
        sl = labels_all[idx]
        sy = years_all[idx]

        for i in range(len(sf) - seq_length):
            seq = sf[i:i+seq_length]
            target = st[i+seq_length]
            label = sl[i+seq_length]
            target_year = sy[i+seq_length]

            if target_year <= 2023:
                train_X.append(seq); train_yr.append(target); train_yc.append(label)
            elif target_year == 2024:
                val_X.append(seq); val_yr.append(target); val_yc.append(label)
            else:
                test_X.append(seq); test_yr.append(target); test_yc.append(label)

    return (np.array(train_X, dtype=np.float32), np.array(train_yr, dtype=np.float32), np.array(train_yc, dtype=np.int64),
            np.array(val_X, dtype=np.float32), np.array(val_yr, dtype=np.float32), np.array(val_yc, dtype=np.int64),
            np.array(test_X, dtype=np.float32), np.array(test_yr, dtype=np.float32), np.array(test_yc, dtype=np.int64))

X_train, yr_train, yc_train, X_val, yr_val, yc_val, X_test, yr_test, yc_test = \
    create_sequences(features_scaled, targets_scaled, labels_all, years_all, station_ids, SEQ_LENGTH)

print(f"\n[FIX-2] 시퀀스 생성 (타겟 = 시퀀스 다음 시점):")
print(f"  학습: {X_train.shape}")
print(f"  검증: {X_val.shape}")
print(f"  테스트: {X_test.shape}")

class ClimateDataset(Dataset):
    def __init__(self, X, yr, yc):
        self.X = torch.from_numpy(X)
        self.yr = torch.from_numpy(yr)
        self.yc = torch.from_numpy(yc)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yr[i], self.yc[i]

BATCH_SIZE = 256
train_loader = DataLoader(ClimateDataset(X_train, yr_train, yc_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(ClimateDataset(X_val, yr_val, yc_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ClimateDataset(X_test, yr_test, yc_test), batch_size=BATCH_SIZE, shuffle=False)

print("\n[2단계] Time Series Transformer + PINN 모델 생성")
print("-"*70)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=3,
                 dim_ff=128, dropout=0.1, num_classes=3, num_reg_targets=3):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        self.attn_pool = nn.Linear(d_model, 1)
        self.reg_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model//2), nn.GELU(),
            nn.Linear(d_model//2, num_reg_targets)
        )
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model//2), nn.GELU(),
            nn.Linear(d_model//2, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        w = torch.softmax(self.attn_pool(x), dim=1)
        x_pool = (x * w).sum(dim=1)
        return self.reg_head(x_pool), self.cls_head(x_pool)

INPUT_DIM = X_train.shape[2]
NUM_CLASSES = len(np.unique(yc_train))

model = TimeSeriesTransformer(
    input_dim=INPUT_DIM, d_model=64, nhead=4, num_layers=3,
    dim_ff=128, dropout=0.15, num_classes=NUM_CLASSES, num_reg_targets=3
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터: {total_params:,}")
print(f"입력: {INPUT_DIM}차원 (타겟 변수 제외), 시퀀스: {SEQ_LENGTH}일")

class PINNLoss(nn.Module):
    def __init__(self, lambda_phys=0.1, year_norm_idx=None, class_weights=None):
        super().__init__()
        self.mse = nn.MSELoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.lambda_phys = lambda_phys
        self.year_norm_idx = year_norm_idx

    def forward(self, reg_pred, cls_pred, reg_target, cls_target, x_input):
        loss_reg = self.mse(reg_pred, reg_target)
        loss_cls = self.ce(cls_pred, cls_target)
        loss_data = loss_reg + 0.5 * loss_cls

        pred_temp = reg_pred[:, 0]

        pred_es = reg_pred[:, 2]
        corr_es = torch.mean(pred_temp * pred_es) - torch.mean(pred_temp) * torch.mean(pred_es)
        loss_magnus = torch.relu(-corr_es)

        loss_bound = (torch.mean(torch.relu(-reg_pred[:, 1] - 3) ** 2) +
                      torch.mean(torch.relu( reg_pred[:, 1] - 3) ** 2))

        if self.year_norm_idx is not None:
            yr = x_input[:, -1, self.year_norm_idx]
            corr_warm = torch.mean(yr * pred_temp) - torch.mean(yr) * torch.mean(pred_temp)
            loss_warming = torch.relu(-corr_warm)
        else:
            loss_warming = torch.tensor(0.0, device=x_input.device)

        loss_physics = loss_magnus + 0.01 * loss_bound + 0.1 * loss_warming
        return loss_data + self.lambda_phys * loss_physics, loss_data, loss_physics

YEAR_NORM_IDX = input_feature_cols.index('year_norm')

_counts = np.bincount(yc_train)
_weights = (1.0 / _counts) * _counts.sum() / len(_counts)
_class_weights = torch.tensor(_weights, dtype=torch.float).to(DEVICE)
print(f"\n클래스 가중치: {dict(enumerate(_weights.round(3)))}")

criterion = PINNLoss(lambda_phys=0.1, year_norm_idx=YEAR_NORM_IDX,
                     class_weights=_class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=7, factor=0.5, min_lr=1e-5)

print("\n[3단계] 모델 학습")
print("-"*70)

EPOCHS = 100
best_val_loss = float('inf')
best_val_acc  = 0.0
best_state    = None
patience_counter = 0
PATIENCE = 10
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(EPOCHS):
    model.train()
    ep_loss, ep_n, correct, total = 0, 0, 0, 0
    for bx, byr, byc in train_loader:
        bx, byr, byc = bx.to(DEVICE), byr.to(DEVICE), byc.to(DEVICE)
        optimizer.zero_grad()
        rp, cp = model(bx)
        loss, ld, lp = criterion(rp, cp, byr, byc, bx)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += ld.item() * bx.size(0)
        ep_n += bx.size(0)
        _, pred = torch.max(cp, 1)
        total += byc.size(0)
        correct += (pred == byc).sum().item()

    train_loss = ep_loss / ep_n
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval()
    vloss, vloss_n, vcorrect, vtotal = 0, 0, 0, 0
    with torch.no_grad():
        for bx, byr, byc in val_loader:
            bx, byr, byc = bx.to(DEVICE), byr.to(DEVICE), byc.to(DEVICE)
            rp, cp = model(bx)
            loss, ld, lp = criterion(rp, cp, byr, byc, bx)
            n = bx.size(0)
            vloss   += ld.item() * n
            vloss_n += n
            _, pred = torch.max(cp, 1)
            vtotal  += n
            vcorrect += (pred == byc).sum().item()

    val_loss = vloss / max(vloss_n, 1)
    val_acc  = 100 * vcorrect / vtotal if vtotal > 0 else 0.0
    scheduler.step(val_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.1f}%")

    if patience_counter >= PATIENCE:
        print(f"\n  [Early Stopping] {PATIENCE}에폭 동안 개선 없음")
        break

print(f"\n  Best Val Loss: {best_val_loss:.4f}  (Acc at that point: {best_val_acc:.2f}%)")

if best_state:
    model.load_state_dict(best_state)

print("\n[4단계] 테스트 평가")
print("-"*70)

model.eval()
all_pred_reg, all_tgt_reg, all_pred_cls, all_tgt_cls = [], [], [], []
with torch.no_grad():
    for bx, byr, byc in test_loader:
        bx = bx.to(DEVICE)
        rp, cp = model(bx)
        _, pred = torch.max(cp, 1)
        all_pred_reg.append(rp.cpu().numpy())
        all_tgt_reg.append(byr.cpu().numpy())
        all_pred_cls.append(pred.cpu().numpy())
        all_tgt_cls.append(byc.cpu().numpy())

if not all_pred_reg:
    raise RuntimeError("테스트 데이터가 없습니다 (2025년 데이터 확인 필요).")
pred_reg = np.concatenate(all_pred_reg)
tgt_reg = np.concatenate(all_tgt_reg)
pred_cls = np.concatenate(all_pred_cls)
tgt_cls = np.concatenate(all_tgt_cls)

test_acc = 100 * np.mean(pred_cls == tgt_cls) if len(pred_cls) > 0 else 0.0
print(f"  기후대 분류 정확도: {test_acc:.2f}%")

pred_reg_orig = scaler_targets.inverse_transform(pred_reg)
tgt_reg_orig = scaler_targets.inverse_transform(tgt_reg)
mae_temp = np.mean(np.abs(pred_reg_orig[:,0] - tgt_reg_orig[:,0]))
mae_rh = np.mean(np.abs(pred_reg_orig[:,1] - tgt_reg_orig[:,1]))
mae_es = np.mean(np.abs(pred_reg_orig[:,2] - tgt_reg_orig[:,2]))
print(f"  온도 예측 MAE: {mae_temp:.2f}°C")
print(f"  습도 예측 MAE: {mae_rh:.2f}%")
print(f"  포화수증기압 MAE: {mae_es:.2f} hPa")

zone_names = ['온대', '냉온대', '아고산대']
print(f"\n{classification_report(tgt_cls, pred_cls, target_names=zone_names)}")

print("[5단계] 시각화")
print("-"*70)

ep = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('한라산 TST+PINN  —  학습 요약', fontsize=15, fontweight='bold')

def _ema(arr, alpha=0.3):
    s = [arr[0]]
    for v in arr[1:]:
        s.append(alpha * v + (1 - alpha) * s[-1])
    return s

val_losses_smooth = _ema(val_losses)

axes[0].plot(ep, val_losses, color=_C['val'], ls='--', alpha=0.2, lw=1.0)
axes[0].plot(ep, train_losses,       label='학습',       color=_C['train'], lw=2)
axes[0].plot(ep, val_losses_smooth,  label='검증 (EMA)', color=_C['val'], ls='--', lw=2)
axes[0].set_xlabel('에포크')
axes[0].set_ylabel('데이터 손실  (MSE + 0.5·CE)')
axes[0].set_title('학습 vs. 검증 손실')
axes[0].legend(framealpha=0.9)

axes[1].plot(ep, train_accs, label='학습', color=_C['train'], lw=2)
axes[1].plot(ep, val_accs,   label='검증', color=_C['val'], ls='--', lw=2)
axes[1].set_xlabel('에포크')
axes[1].set_ylabel('정확도 (%)')
axes[1].set_title('기후대 분류 정확도')
axes[1].set_ylim(0, 105)
axes[1].legend(framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history_fixed.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"  저장: training_history_fixed.png")
_show(f'{OUTPUT_DIR}/training_history_fixed.png')

cm     = confusion_matrix(tgt_cls, pred_cls)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
cb = fig.colorbar(im, fraction=0.046, pad=0.04)
cb.set_label('행 비율 (%)', fontsize=11)
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(zone_names, fontsize=11)
ax.set_yticklabels(zone_names, fontsize=11)
ax.set_xlabel('예측 레이블')
ax.set_ylabel('실제 레이블')
ax.set_title('혼동 행렬')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)',
                ha='center', va='center', fontsize=12,
                color='white' if cm_pct[i, j] > 50 else '#1A1A1A')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_fixed.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"  저장: confusion_matrix_fixed.png")
_show(f'{OUTPUT_DIR}/confusion_matrix_fixed.png')

print("\n[6단계] 고도별 기후대 예측")
print("-"*70)

def predict_altitude(model, altitude_m, scaler_features, scaler_targets,
                     month=None, year=None):
    model.eval()
    pred_year = year if year is not None else 2025

    year_trend_val = (pred_year - BASELINE_YEAR) * WARMING_RATE
    year_norm_val  = min(1.0, (pred_year - BASELINE_YEAR) / _year_span)

    base_temp = 15.0
    lapse = -0.58 / 100
    est_temp = base_temp + lapse * altitude_m + year_trend_val
    if month:
        est_temp += 12 * math.cos(2 * math.pi * (month - 7) / 12)

    est_rh = min(95, 60 + (altitude_m / 1950) * 35)

    seq = []
    for d in range(SEQ_LENGTH):
        m = month if month else ((d * 12 // SEQ_LENGTH) % 12) + 1
        doy = (m - 1) * 30 + d + 1
        tv = est_temp + np.random.normal(0, 1.5)
        tr = max(0.5, 5 + np.random.normal(0, 1))
        tmin, tmax = tv - tr / 2, tv + tr / 2
        rh_min_v = max(1, est_rh - 15 + np.random.normal(0, 3))
        rh_max_v = min(100, est_rh + 15 + np.random.normal(0, 3))
        rh_m = (rh_min_v + rh_max_v) / 2
        denom = tv + 237.3
        es_t = 6.1078 * math.exp((17.27 * tv) / denom) if denom != 0 else 6.1078
        ea = max(0.01, (rh_m / 100) * es_t)
        ln_r = math.log(ea / 6.1078)
        dp = (237.3 * ln_r) / (17.27 - ln_r)
        ws = np.random.exponential(7)
        rain = np.random.exponential(2) if np.random.random() < 0.3 else 0
        wd = np.random.uniform(0, 360)
        seq.append([
            tmin, tmax, tr, rh_min_v, rh_max_v, dp,
            ws, rain,
            math.sin(math.radians(wd)), math.cos(math.radians(wd)),
            math.sin(2 * math.pi * m / 12), math.cos(2 * math.pi * m / 12),
            math.sin(2 * math.pi * doy / 365), math.cos(2 * math.pi * doy / 365),
            year_norm_val, year_trend_val,
        ])

    seq_arr = scaler_features.transform(np.array(seq, dtype=np.float32))
    inp = torch.from_numpy(seq_arr).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        rp, cp = model(inp)
        reg_orig = scaler_targets.inverse_transform(rp.cpu().numpy())[0]
        probs = torch.softmax(cp, dim=-1).cpu().numpy()[0]
        zone = int(np.argmax(probs))

    return {
        'altitude': altitude_m,
        'year': pred_year,
        'zone': zone,
        'zone_name': {0: '온대', 1: '냉온대', 2: '아고산대'}[zone],
        'probs': probs,
        'pred_temp': float(reg_orig[0]),
        'pred_rh':   float(reg_orig[1]),
        'pred_es':   float(reg_orig[2]),
    }

np.random.seed(SEED)
results = []
print(f"\n{'고도(m)':<8} {'기후대':<18} {'온도(°C)':<10} {'습도(%)':<10} {'확률'}")
print("-"*65)
for alt in range(200, 2001, 100):
    r = predict_altitude(model, alt, scaler_features, scaler_targets, year=2025)
    results.append(r)
    max_p = max(r['probs'])
    print(f"{alt:<8} {r['zone_name']:<18} {r['pred_temp']:<10.1f} {r['pred_rh']:<10.1f} {max_p:.3f}")

alts  = [r['altitude']  for r in results]
temps = [r['pred_temp'] for r in results]
zones = [r['zone']      for r in results]
p0    = [r['probs'][0]  for r in results]
p1    = [r['probs'][1]  for r in results]
p2    = [r['probs'][2]  for r in results]
p01   = [a + b for a, b in zip(p0, p1)]

_zone_color = {0: _C['temperate'], 1: _C['cool'], 2: _C['subalpine']}
_zone_label = {0: '온대 (<1200 m)',
               1: '냉온대 (1200–1500 m)',
               2: '아고산대 (>1500 m)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('한라산 기후대 단면도  (2025년)', fontsize=15, fontweight='bold')

for a, t, z in zip(alts, temps, zones):
    axes[0].scatter(t, a, c=_zone_color[z], s=80, zorder=5,
                    edgecolors='white', linewidths=0.5)
axes[0].plot(temps, alts, color='#888888', lw=1.2, alpha=0.5, zorder=4)
axes[0].axhline(1200, color='#E65100', ls='--', lw=1.5, alpha=0.8, label='경계 1200 m')
axes[0].axhline(1500, color='#6A1B9A', ls='--', lw=1.5, alpha=0.8, label='경계 1500 m')
for z_id in [0, 1, 2]:
    axes[0].scatter([], [], c=_zone_color[z_id], s=70, label=_zone_label[z_id])
axes[0].set_xlabel('예측 온도 (°C)')
axes[0].set_ylabel('고도 (m)')
axes[0].set_title('온도 vs. 고도')
axes[0].legend(fontsize=10, framealpha=0.9)

axes[1].fill_betweenx(alts, 0,   p0,  alpha=0.7, color=_C['temperate'], label='온대')
axes[1].fill_betweenx(alts, p0,  p01, alpha=0.7, color=_C['cool'],      label='냉온대')
axes[1].fill_betweenx(alts, p01, 1,   alpha=0.7, color=_C['subalpine'], label='아고산대')
axes[1].axhline(1200, color='#E65100', ls='--', lw=1.5, alpha=0.8)
axes[1].axhline(1500, color='#6A1B9A', ls='--', lw=1.5, alpha=0.8)
axes[1].set_xlabel('확률')
axes[1].set_ylabel('고도 (m)')
axes[1].set_title('기후대 확률')
axes[1].set_xlim(0, 1)
axes[1].legend(fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/altitude_climate_profile_fixed.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"\n  저장: altitude_climate_profile_fixed.png")
_show(f'{OUTPUT_DIR}/altitude_climate_profile_fixed.png')

print("\n[6.5단계] 지구온난화 시나리오별 고도-온도 프로파일")
print("-"*70)

scenario_years = {
    '현재 (2025)'              : 2025,
    '근미래 (2050, SSP2-4.5)' : 2050,
    '중기 (2075, SSP5-8.5)'   : 2075,
    '말기 (2100, SSP5-8.5)'   : 2100,
}

IPCC_DELTA = {
    '현재 (2025)'              : 0.0,
    '근미래 (2050, SSP2-4.5)' : 1.2,
    '중기 (2075, SSP5-8.5)'   : 2.8,
    '말기 (2100, SSP5-8.5)'   : 4.8,
}

_EDW_ALT_MIN = 200
_EDW_ALT_MAX = 1950

def edw_factor(altitude_m):
    t = max(0.0, min(1.0, (altitude_m - _EDW_ALT_MIN) / (_EDW_ALT_MAX - _EDW_ALT_MIN)))
    return 1.0 + 0.35 * t

altitudes_w = list(range(200, 2001, 100))
np.random.seed(SEED)
baseline_preds = [predict_altitude(model, alt, scaler_features, scaler_targets, year=2025)
                  for alt in altitudes_w]
baseline_temp = {r['altitude']: r['pred_temp'] for r in baseline_preds}

warming_results = {}
for label, yr in scenario_years.items():
    delta = IPCC_DELTA[label]
    preds = []
    for r in baseline_preds:
        alt = r['altitude']
        t   = baseline_temp[alt] + delta * edw_factor(alt)
        preds.append({**r, 'pred_temp': t, 'year': yr})
    warming_results[label] = preds

scenario_labels_w = list(scenario_years.keys())
zone_display      = {0: '온대', 1: '냉온대', 2: '아고산'}
col_w = 8

def _yr_tag(lbl):
    return lbl.split('(')[1].split(',')[0].split(')')[0].strip()

header = f"{'고도(m)':<7} {'기후대':<8}"
for lbl in scenario_labels_w:
    header += f" {_yr_tag(lbl):>{col_w}}"
header += f"  {'Δ25→2100':>9}"
print("\n" + header)
print("-" * len(header))

temp_grid = {}
for alt_idx, alt in enumerate(altitudes_w):
    temps = [warming_results[lbl][alt_idx]['pred_temp'] for lbl in scenario_labels_w]
    temp_grid[alt] = temps
    zone_id = classify_climate_by_altitude(alt)
    delta   = temps[-1] - temps[0]
    row = f"{alt:<7} {zone_display[zone_id]:<8}"
    for t in temps:
        row += f" {t:>{col_w}.1f}"
    row += f"  {delta:>+9.1f}°C"
    print(row)

print(f"\n[ 기후대별 평균 온도 ]")
hdr2 = f"{'기후대':<10}"
for lbl in scenario_labels_w:
    hdr2 += f" {_yr_tag(lbl):>{col_w}}"
hdr2 += f"  {'Δ25→2100':>9}"
print(hdr2)
print("-" * len(hdr2))

zone_avg = {}
for z_id, z_nm in {0: '온대', 1: '냉온대', 2: '아고산대'}.items():
    z_alts = [alt for alt in altitudes_w if classify_climate_by_altitude(alt) == z_id]
    if not z_alts:
        continue
    avg_per_scen = [np.mean([temp_grid[alt][i] for alt in z_alts])
                    for i in range(len(scenario_labels_w))]
    zone_avg[z_id] = avg_per_scen
    delta = avg_per_scen[-1] - avg_per_scen[0]
    row = f"{z_nm:<10}"
    for t in avg_per_scen:
        row += f" {t:>{col_w}.1f}"
    row += f"  {delta:>+9.1f}°C"
    print(row)

fig = plt.figure(figsize=(17, 9))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.36,
                        left=0.07, right=0.97, top=0.89, bottom=0.09)
ax_prof  = fig.add_subplot(gs[:, 0])
ax_delta = fig.add_subplot(gs[0, 1])
ax_zone  = fig.add_subplot(gs[1, 1])

fig.suptitle('한라산 지구온난화 시나리오 분석  (IPCC AR6 SSP)', fontsize=15, fontweight='bold')

ax_prof.axhspan(  0, 1200, color='#E8F5E9', alpha=0.45, zorder=0)
ax_prof.axhspan(1200, 1500, color='#BBDEFB', alpha=0.45, zorder=0)
ax_prof.axhspan(1500, 2200, color='#EDE7F6', alpha=0.45, zorder=0)

for (label, preds), col in zip(warming_results.items(), _C['scenarios']):
    w_temps = [r['pred_temp'] for r in preds]
    w_alts  = [r['altitude']  for r in preds]
    ax_prof.plot(w_temps, w_alts, marker='o', ms=4, lw=2.2,
                 label=label, color=col, zorder=5)

ax_prof.axhline(1200, color='#E65100', ls='--', lw=1.3, alpha=0.8, zorder=4)
ax_prof.axhline(1500, color='#6A1B9A', ls='--', lw=1.3, alpha=0.8, zorder=4)

ax_prof.text(0.03, 0.275, '온대\n낙엽활엽수림대',
             transform=ax_prof.transAxes, fontsize=9,
             color='#2E7D32', va='center', alpha=0.75)
ax_prof.text(0.03, 0.625, '냉온대\n침엽수림대',
             transform=ax_prof.transAxes, fontsize=9,
             color='#1565C0', va='center', alpha=0.75)
ax_prof.text(0.03, 0.85,  '아고산대',
             transform=ax_prof.transAxes, fontsize=9,
             color='#6A1B9A', va='center', alpha=0.75)

ax_prof.set_xlabel('예측 온도 (°C)')
ax_prof.set_ylabel('고도 (m)')
ax_prof.set_title('고도-온도 단면도\n(기후대 배경 포함)', fontsize=12)
ax_prof.set_ylim(100, 2100)
ax_prof.legend(fontsize=9, framealpha=0.9, loc='lower right')

baseline_t = {alt: temp_grid[alt][0] for alt in altitudes_w}

for i, (label, preds) in enumerate(list(warming_results.items())[1:], 1):
    deltas = [temp_grid[alt][i] - baseline_t[alt] for alt in altitudes_w]
    ax_delta.plot(deltas, altitudes_w, marker='o', ms=4, lw=2,
                  label=label, color=_C['scenarios'][i], zorder=5)

ax_delta.axhline(1200, color='#E65100', ls='--', lw=1.2, alpha=0.7, zorder=4)
ax_delta.axhline(1500, color='#6A1B9A', ls='--', lw=1.2, alpha=0.7, zorder=4)
ax_delta.axvline(0, color='gray', ls='-', lw=0.8, alpha=0.4)
ax_delta.set_xlabel('온도 변화량 ΔT (°C,  2025년 대비)')
ax_delta.set_ylabel('고도 (m)')
ax_delta.set_title('고도별 온난화 증분\n(2025년 기준)', fontsize=12)
ax_delta.set_ylim(100, 2100)
ax_delta.legend(fontsize=8.5, framealpha=0.9)

zone_ids_ordered = [z for z in [0, 1, 2] if z in zone_avg]
_zone_bar_names  = {0: '온대\n낙엽활엽수림대', 1: '냉온대\n침엽수림대', 2: '아고산대'}
zone_names_bar   = [_zone_bar_names[z] for z in zone_ids_ordered]
n_zones = len(zone_ids_ordered)
n_scen  = len(scenario_labels_w)
bar_w   = 0.18
x_pos   = np.arange(n_zones)

for i, (label, col) in enumerate(zip(scenario_labels_w, _C['scenarios'])):
    vals   = [zone_avg[z_id][i] for z_id in zone_ids_ordered]
    offset = (i - (n_scen - 1) / 2) * bar_w
    bars   = ax_zone.bar(x_pos + offset, vals, width=bar_w,
                         label=_yr_tag(label), color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax_zone.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.05,
                     f'{v:.1f}', ha='center', va='bottom', fontsize=8)

ax_zone.set_xticks(x_pos)
ax_zone.set_xticklabels(zone_names_bar, fontsize=10)
ax_zone.set_ylabel('평균 예측 온도 (°C)')
ax_zone.set_title('기후대별 평균 온도\n(시나리오 비교)', fontsize=12)
ax_zone.legend(fontsize=8.5, framealpha=0.9, loc='upper right', title='연도')

plt.savefig(f'{OUTPUT_DIR}/warming_scenarios.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"\n  저장: warming_scenarios.png")
_show(f'{OUTPUT_DIR}/warming_scenarios.png')

print("\n[7단계] 모델 저장")
print("-"*70)

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'input_dim': INPUT_DIM, 'd_model': 64, 'nhead': 4, 'num_layers': 3,
        'dim_ff': 128, 'dropout': 0.15, 'num_classes': NUM_CLASSES, 'num_reg_targets': 3,
        'baseline_year': BASELINE_YEAR, 'warming_rate': WARMING_RATE,
    },
    'scaler_features': scaler_features,
    'scaler_targets': scaler_targets,
    'input_feature_cols': input_feature_cols,
    'target_cols': target_cols,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'mae_temp': mae_temp, 'mae_rh': mae_rh, 'mae_es': mae_es,
}, f'{OUTPUT_DIR}/hallasan_model_fixed.pth')
print(f"  저장: hallasan_model_fixed.pth")

print("\n" + "="*70)
print("  최종 요약 (데이터 누수 수정 버전)")
print("="*70)
print(f"""
  모델: Time Series Transformer + PINN
  ├── 구조: 3-layer Transformer Encoder (d=64, h=4)
  ├── 물리 제약: 마그누스 공식, 습도 경계, 지구온난화 트렌드
  ├── 시퀀스: {SEQ_LENGTH}일, 피처: {INPUT_DIM}개 (타겟 제외)
  ├── 파라미터: {total_params:,}
  │
  ├── [성능]
  │   ├── 기후대 분류 정확도: {test_acc:.2f}%
  │   ├── 온도 MAE: {mae_temp:.2f}°C
  │   ├── 습도 MAE: {mae_rh:.2f}%
  │   └── 포화수증기압 MAE: {mae_es:.2f} hPa
  │
  └── [기후대]
      ├── Class 0: 온대 낙엽활엽수림대 (< 1200m)
      ├── Class 1: 냉온대 침엽수림대 (1200-1500m)
      └── Class 2: 아고산대 (> 1500m)
""")
print("="*70)
print("  완료!")
print("="*70)

  실행 환경: Google Colab | 디바이스: cpu
  한라산 고도별 기후대 예측 AI (데이터 누수 수정 버전)
  Time Series Transformer + PINN

[1단계] 데이터 로드 및 전처리
----------------------------------------------------------------------


FileNotFoundError: [Errno 2] No such file or directory: '/content/halla_aws_extreme_all.csv'